In [29]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

idf_threshold: float = 1.5
texts = ["figure 1 an overview of our LeakGAN text generation framework while the generator be responsible to generate the next word the discriminator adversarially judge the generate sentence once it be complete the chief novelty lie in that unlike conventional adversarial training during the process the discriminator reveal its internal state ( f t in order to guide the generator more informatively and frequently ( methodology section for more detail ) section relate work generate text that mimic human 's expression have be study for poem generation [ image captioning dialogue system machine translation [ propose a recurent neural network ( base generative model to use the human - generate text where at each step the model try to predict the next word give previous real word sequence and be train in a supervise fashion a common difficulty of all supervise generative model be that it be hard to design an appropriate differentiable lowbias metric to evaluate the output of the generator which inspire the adversarial train mechanism [ propose generative adversarial net ( to generate continuous datum like image GAN introduce a minimax game between a generative model and a discriminative model where the discriminator can be view as the dynamically - update evaluation metric to guide the tune of the generate datum to apply gan to text generation [ propose SeqGAN that model the text generation as a sequential decision make process and train the generative model with policy gradient method MaliGAN [ modifie the orginal GAN objective and propose a set of train technique to reduce the potential variance"]

vectorizer = TfidfVectorizer(use_idf=True, norm=None, smooth_idf=False)
X = vectorizer.fit_transform(texts)
idf = vectorizer.idf_
feature_names = vectorizer.get_feature_names_out()
    
# 2. Фильтруем слова с низким IDF (слишком распространенные)
words_to_keep = {
    word for word, score in zip(feature_names, idf) 
    if score >= idf_threshold
}
    
# 3. Применяем фильтр к исходным текстам
filtered_texts = []
for text in texts:
    words = text.split()
    print(words)
    filtered_words = [w for w in words if w in words_to_keep]
    print(filtered_words)
    filtered_texts.append(' '.join(filtered_words))

['figure', '1', 'an', 'overview', 'of', 'our', 'LeakGAN', 'text', 'generation', 'framework', 'while', 'the', 'generator', 'be', 'responsible', 'to', 'generate', 'the', 'next', 'word', 'the', 'discriminator', 'adversarially', 'judge', 'the', 'generate', 'sentence', 'once', 'it', 'be', 'complete', 'the', 'chief', 'novelty', 'lie', 'in', 'that', 'unlike', 'conventional', 'adversarial', 'training', 'during', 'the', 'process', 'the', 'discriminator', 'reveal', 'its', 'internal', 'state', '(', 'f', 't', 'in', 'order', 'to', 'guide', 'the', 'generator', 'more', 'informatively', 'and', 'frequently', '(', 'methodology', 'section', 'for', 'more', 'detail', ')', 'section', 'relate', 'work', 'generate', 'text', 'that', 'mimic', 'human', "'s", 'expression', 'have', 'be', 'study', 'for', 'poem', 'generation', '[', 'image', 'captioning', 'dialogue', 'system', 'machine', 'translation', '[', 'propose', 'a', 'recurent', 'neural', 'network', '(', 'base', 'generative', 'model', 'to', 'use', 'the', 'human'

In [2]:
filtered_texts

['figure an overview of our text generation framework while the generator be responsible to generate the next word the discriminator adversarially judge the generate sentence once it be complete the chief novelty lie in that unlike conventional adversarial training during the process the discriminator reveal its internal state in order to guide the generator more informatively and frequently methodology section for more detail section relate work generate text that mimic human expression have be study for poem generation image captioning dialogue system machine translation propose recurent neural network base generative model to use the human generate text where at each step the model try to predict the next word give previous real word sequence and be train in supervise fashion common difficulty of all supervise generative model be that it be hard to design an appropriate differentiable lowbias metric to evaluate the output of the generator which inspire the adversarial train mechanis

In [31]:
texts = ["figure 1 an overview of our LeakGAN text generation framework while the generator be responsible to generate the next word the discriminator adversarially judge the generate sentence once it be complete the chief novelty lie in that unlike conventional adversarial training during the process the discriminator reveal its internal state ( f t in order to guide the generator more informatively and frequently ( methodology section for more detail ) section relate work generate text that mimic human 's expression have be study for poem generation [ image captioning dialogue system machine translation [ propose a recurent neural network ( base generative model to use the human - generate text where at each step the model try to predict the next word give previous real word sequence and be train in a supervise fashion a common difficulty of all supervise generative model be that it be hard to design an appropriate differentiable lowbias metric to evaluate the output of the generator which inspire the adversarial train mechanism [ propose generative adversarial net ( to generate continuous datum like image GAN introduce a minimax game between a generative model and a discriminative model where the discriminator can be view as the dynamically - update evaluation metric to guide the tune of the generate datum to apply gan to text generation [ propose SeqGAN that model the text generation as a sequential decision make process and train the generative model with policy gradient method MaliGAN [ modifie the orginal GAN objective and propose a set of train technique to reduce the potential variance"]

In [30]:
def dymamic_filter_uniform_words(
    texts: List[str],
    initial_idf_threshold: float = 1.5,
    min_idf_threshold: float = 0.5,
    step_reduction: float = 0.2,
    min_step: float = 0.05,
    target_words_per_doc: int = 3
) -> Tuple[List[str], float]:
    """
    Фильтрует слова с низкой дискриминативной способностью (IDF) с динамической настройкой порога.
    
    Параметры:
        texts: список документов
        initial_idf_threshold: начальное значение порога IDF
        min_idf_threshold: минимально допустимое значение IDF
        step_reduction: шаг уменьшения порога при отсутствии результатов
        min_step: минимальный шаг уменьшения
        target_words_per_doc: целевое минимальное количество слов в документе
    
    Возвращает:
        Кортеж: (отфильтрованные тексты, использованное значение IDF)
    """
    # 1. Подготовка данных
    original_words = [text.split() for text in texts]
    lemmatized_texts = [
        ' '.join(lemmatize_en(word) for word in words)
        for words in original_words
    ]
    
    # 2. Вычисление IDF
    vectorizer = TfidfVectorizer(use_idf=True, norm=None, smooth_idf=False)
    X = vectorizer.fit_transform(lemmatized_texts)
    idf = vectorizer.idf_
    feature_names = vectorizer.get_feature_names_out()
    
    # 3. Динамический подбор порога
    current_threshold = initial_idf_threshold
    current_step = step_reduction
    best_result = None
    best_threshold = current_threshold
    
    while current_threshold >= min_idf_threshold:
        # Фильтрация слов
        words_to_keep = {
            word for word, score in zip(feature_names, idf) 
            if score >= current_threshold
        }
        
        # Применение фильтра
        filtered_texts = []
        valid_docs_count = 0
        
        for text in lemmatized_texts:
            words = text.split()
            filtered_words = [w for w in words if w in words_to_keep]
            filtered_text = ' '.join(filtered_words)
            filtered_texts.append(filtered_text)
            
            if len(filtered_words) >= target_words_per_doc:
                valid_docs_count += 1
        
        # Проверка качества фильтрации
        doc_coverage = valid_docs_count / len(texts)
        
        # Сохраняем лучший результат
        if best_result is None or doc_coverage > best_result[0]:
            best_result = (doc_coverage, filtered_texts, current_threshold)
        
        # Критерий остановки
        if doc_coverage >= 0.8:  # 80% документов с достаточным количеством слов
            break
            
        # Уменьшаем порог
        current_threshold -= current_step
        current_step = max(current_step * 0.8, min_step)
    
    # Возвращаем лучший найденный результат
    if best_result is None:
        logger.warning("Не удалось найти подходящий порог IDF. Возвращаю исходные тексты.")
        return texts, 0.0
    
    logger.info(f"Использованный порог IDF: {best_result[2]:.2f}, покрытие документов: {best_result[0]*100:.1f}%")
    return best_result[1], best_result[2]

In [34]:
filt_results = dymamic_filter_uniform_words(texts = texts)
filt_results[0]

['figure an overview of our text generation framework while the generator be responsible to generate the next word the discriminator adversarially judge the generate sentence once it be complete the chief novelty lie in that unlike conventional adversarial train during the process the discriminator reveal its internal state in order to guide the generator more informatively and frequently methodology section for more detail section relate work generate text that mimic human expression have be study for poem generation image captioning dialogue system machine translation propose recurent neural network base generative model to use the human generate text where at each step the model try to predict the next word give previous real word sequence and be train in supervise fashion common difficulty of all supervise generative model be that it be hard to design an appropriate differentiable lowbias metric to evaluate the output of the generator which inspire the adversarial train mechanism p

------------------------------------

In [3]:
import sys
sys.path.append("/global_functions")
from global_functions.global_functions import *

------------------------------------
---> Choosing resource
---> Result of torch.cuda.is_available():
True
---> Final resource
-----> CUDA: 1
-----> device: cuda:1
------------------------------------


/home/pmartynyuk/UnTIE project/untie_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
loading file vocab.txt
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
loading file tokenizer.json
loading file chat_template.jinja
loading configuration file /home/pmartynyuk/UnTIE project/scripts/models_processing/models/eng_sentence_transformer_model/config.json
Model config BertConfig {
  "architectures": [
    "BertModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_atte

In [4]:
# Датасет
datasets_path = "../datasets"
name = "scirex_structured.json"

# Результаты
datasets_results_path = "../datasets_results"
res_name = "results_1.json"

# Модель документа
model_params_path = "../model_params"
model_name = "scart_init_model.json"
#----------------------------

# Загрузка датасета
df_loaded = load_dataframe_from_json(f"{datasets_path}/{name}")
df_loaded = replace_underscores_in_array_column(df=df_loaded,
                                        column_name="tasks")

In [5]:
# Чтение параметров модели
df_model_params = pd.read_json(f"{model_params_path}/{model_name}")

field_name = df_model_params["fields"][0]["field_name"]
questions = df_model_params["fields"][0]["questions"]
keywords = df_model_params["fields"][0]["keywords"]
text = df_model_params["fields"][0]["text"]

In [6]:
num_of_sample = 0

In [7]:
# Тематический аспект и данные для фильтрации
them_aspect = ThematicAspect(field_name, convert_into_question_class(questions))
filtering_set = FilteringSet(text, keywords)

text_of_doc = df_loaded["original_text"][num_of_sample]
reference_answers = df_loaded["tasks_cleaned"][num_of_sample]

res1 = extract_short_answer(text=text_of_doc,
                    them_asp=them_aspect, 
                    filter_mode=FilterMode.NO_FILTER,
                    filter_set=filtering_set,
                    agg_mode=AggMode.NO_AGG)

res2 = extract_keywords(them_asp=them_aspect,
                            reference_answer=reference_answers[0])

Device set to use cuda:1
Disabling tokenizer parallelism, we're using DataLoader multithreading already
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [8]:
all_texts = [chunk.text for chunk in res1['used_chunks']]
all_texts

['Full - Resolution Residual Networks for Semantic Segmentation in Street Scenes section: Abstract Semantic image segmentation is an essential component of modern autonomous driving systems, as an accurate understanding of the surrounding scene is crucial to navigation and action planning. Current state - of - the - art approaches in semantic image segmentation rely on pretrained networks that were initially developed for classifying images as a whole. While these networks exhibit outstanding recognition performance (i.e., what is visible? ), they lack localization accuracy (i.e., where precisely is something located? ). Therefore, additional processing steps have to be performed in order to obtain pixel - accurate segmentation masks at the full image resolution. To alleviate this problem we propose a novel ResNet - like architecture that exhibits strong localization and recognition performance. We combine multi - scale context with pixel - level accuracy by using two processing stream

In [9]:
target_texts = [chunk.text for chunk in res2['analysis']['chunks']]
target_texts

['Semantic image segmentation [reference ][ reference ][ reference ][ reference ][ reference], the task of assigning a set of predefined class labels to image pixels, is an important tool for modeling the complex relationships of the semantic entities usually found in street scenes, such as cars, pedestrians, road, or sidewalks. In automotive scenarios it is used in various ways, e.g. as a pre - processing step to discard image regions that are unlikely to contain objects of interest [reference ][ reference], to improve object detection [reference ][ reference ][ reference ][ reference]. Example output and the abstract structure of our fullresolution residual network. The network has two processing streams. The residual stream (blue) stays at the full image resolution, the pooling stream (red) undergoes a sequence of pooling and unpooling operations. The two processing streams are coupled using full - resolution residual units [reference] or in combination with 3D scene geometry [refer

In [10]:
import numpy as np
from typing import List, Tuple
from sklearn.feature_extraction.text import TfidfVectorizer
import logging

logger = logging.getLogger(__name__)

def filter_characteristic_words(
    target_texts: List[str],
    background_texts: List[str],
    initial_idf_threshold: float = 1.5,
    min_idf_threshold: float = 0.5,
    step_reduction: float = 0.2,
    min_step: float = 0.05,
    target_words_per_doc: int = 3
) -> Tuple[List[str], float]:
    """
    Фильтрует слова, характерные для целевых текстов на фоне общей коллекции.
    
    Параметры:
        target_texts: тексты, которые нужно охарактеризовать
        background_texts: фоновая коллекция текстов для сравнения
        initial_idf_threshold: начальное значение порога IDF
        min_idf_threshold: минимально допустимое значение IDF
        step_reduction: шаг уменьшения порога
        min_step: минимальный шаг уменьшения
        target_words_per_doc: минимальное количество слов в документе
    
    Возвращает:
        Кортеж: (отфильтрованные тексты, использованное значение IDF)
    """
    # 1. Подготовка данных (ОСНОВНОЕ ИЗМЕНЕНИЕ - добавляем фоновые тексты)
    all_texts = background_texts
    target_indices = [all_texts.index(text) for text in target_texts]
    
    # Лемматизация для всех текстов
    lemmatized_all = [
        ' '.join(lemmatize_en(word) for word in text.split())
        for text in all_texts
    ]
    lemmatized_targets = [lemmatized_all[i] for i in target_indices]

    # 2. Вычисление IDF на объединенной коллекции (ОСНОВНОЕ ИЗМЕНЕНИЕ)
    vectorizer = TfidfVectorizer(use_idf=True, norm=None, smooth_idf=False)
    X = vectorizer.fit_transform(lemmatized_all)
    idf = vectorizer.idf_
    feature_names = vectorizer.get_feature_names_out()

    # 3. Вычисление относительной важности слов (НОВЫЙ БЛОК)
    target_word_counts = np.sum(X[target_indices] > 0, axis=0).A1
    background_word_counts = np.sum(X[len(target_texts):] > 0, axis=0).A1
    word_ratios = (target_word_counts + 1) / (background_word_counts + 1)  # +1 для сглаживания

    # 4. Комбинированный критерий (НОВЫЙ БЛОК)
    combined_scores = idf * word_ratios
    
    # 5. Адаптивный подбор порога (МОДИФИЦИРОВАННЫЙ БЛОК)
    current_threshold = initial_idf_threshold
    best_result = None
    
    while current_threshold >= min_idf_threshold:
        words_to_keep = {
            word for word, score in zip(feature_names, combined_scores)
            if score >= current_threshold
        }
        
        filtered_texts = []
        valid_docs = 0
        
        for text in lemmatized_targets:
            words = text.split()
            filtered = [w for w in words if w in words_to_keep]
            filtered_texts.append(' '.join(filtered))
            valid_docs += len(filtered) >= target_words_per_doc
        
        coverage = valid_docs / len(target_texts)
        
        if best_result is None or coverage > best_result[0]:
            best_result = (coverage, filtered_texts, current_threshold)
            
        if coverage >= 0.8:
            break
            
        current_threshold -= max(step_reduction * (1 - coverage), min_step)
    
    return best_result[1:] if best_result else (target_texts, 0.0)

In [11]:
results = filter_characteristic_words(target_texts=target_texts,
                                background_texts=all_texts)

In [12]:
results[0]

['task assign predefine important tool complex relationship entity usually car pedestrian road sidewalk automotive scenario various discard region unlikely interest detection example abstract structure fullresolution stay undergoe couple combination 3d geometry those application precise region therefore pursue goal precise current employ form convolutional take probability rely already prove successful variant start target task auxiliary task yield scratch target application',
 'abstract essential component modern autonomous drive system accurate understand surround navigation action planning current rely pretraine initially develop classify whole exhibit outstanding what visible lack accuracy precisely something locate therefore accurate mask alleviate like exhibit strong multi context accuracy carry enable precise segment undergoe robust couple pretraine intersection union 71 introduction recent year interest self drive car driver assistance system aspect autonomous drive acquire com

In [13]:
keywords_1 = find_keywords(SentenceTokenizerSingleton().tokenizer,
                                sentences_candidat=results[0], sentences_reference =results[0], 
                                collocation_len=1, num_of_keywords=30) 

In [14]:
keywords_1

['year',
 'pursue',
 'usually',
 'undergoe',
 'scratch',
 'current',
 'start',
 'application',
 'probability',
 'precise',
 'relationship',
 'target',
 'road',
 'employ',
 'classify',
 'self',
 'assign',
 'navigation',
 'intersection',
 'prove',
 'fullresolution',
 'applicationabstract',
 'modern',
 'predefine',
 'pretraine',
 'drive',
 'driver',
 'geometry',
 'car',
 'automotive']

In [15]:
keys = filter_uniform_words(texts=target_texts)
keys

['automotive scenario it various way pre discard region unlikely contain object improve object detection example output structure fullresolution stay unpoole unit combination 3d geometry many those application require region work pursue goal high quality all employ some form fully convolutional take input output probability map each many paper already prove successful classification variant start from pre train net large number weight target can pre auxiliary classification reduce train time often yield superior compare train from scratch limit amount datum target application',
 'section essential component modern autonomous drive system accurate understand surround crucial navigation action planning pretraine initially develop classify whole while these exhibit outstanding recognition what visible they lack localization accuracy precisely something locate additional perform order obtain accurate mask alleviate problem propose novel like exhibit strong localization recognition combine 

In [16]:
keywords_2 = find_keywords(SentenceTokenizerSingleton().tokenizer,
                                sentences_candidat=keys, sentences_reference =keys, 
                                collocation_len=1, num_of_keywords=30) 

In [17]:
keywords_2

['goal',
 'increase',
 'start',
 'output',
 'navigation',
 'accurate',
 'improve',
 'unpoole',
 'pursue',
 'combine',
 'target',
 'train',
 'self',
 'probability',
 'superior',
 'novel',
 'scratch',
 'prove',
 'automotive',
 'planning',
 'score',
 'accuracy',
 'yield',
 'driver',
 'applicationsection',
 'drive',
 'modern',
 'fullresolution',
 'pretraine',
 'geometry']

In [22]:
def sort_words_by_cosine_similarity(words: list[str], 
                                    text: str,
                                    model = SentenceTokenizerSingleton().tokenizer
                                    ) -> list[tuple[str, float]]:
    """
    Сортирует слова по косинусной близости их эмбеддингов к эмбеддингу текста.
    
    Параметры:
        words: список слов для сортировки
        text: текст, к которому вычисляется близость
    
    Возвращает:
        Список кортежей (слово, оценка_близости), отсортированный по убыванию близости
    """
    if not words or not text.strip():
        return []
    
    # Получаем эмбеддинги (оставляем их на том же устройстве, что и модель)
    text_embedding = model.encode(text, convert_to_tensor=True)  # (N,)
    word_embeddings = model.encode(words, convert_to_tensor=True)  # (M, N)

    # Добавляем размерность для текста (1, N) и вычисляем близость
    similarities = cosine_similarity(
        text_embedding.unsqueeze(0),  # (1, N)
        word_embeddings,              # (M, N)
        dim=1                         # Сравниваем по последней оси
    )

    # Преобразуем в Python-объекты и сортируем
    scored_words = [
        (word, score.item()) 
        for word, score in zip(words, similarities)
    ]
    scored_words.sort(key=lambda x: x[1], reverse=True)

    return scored_words

In [26]:
filt_res_1 = sort_words_by_cosine_similarity(words = keywords_1, 
                                    text = text)
filt_res_1

[('applicationabstract', 0.5271218419075012),
 ('application', 0.5040464401245117),
 ('classify', 0.49875959753990173),
 ('pretraine', 0.4902573227882385),
 ('undergoe', 0.461254358291626),
 ('pursue', 0.44221925735473633),
 ('target', 0.43687745928764343),
 ('prove', 0.4239189624786377),
 ('employ', 0.4224444627761841),
 ('assign', 0.41466766595840454),
 ('relationship', 0.40681931376457214),
 ('geometry', 0.3997527062892914),
 ('navigation', 0.3869742751121521),
 ('predefine', 0.37276017665863037),
 ('driver', 0.3713556230068207),
 ('start', 0.35531628131866455),
 ('current', 0.3547686040401459),
 ('self', 0.35128113627433777),
 ('probability', 0.31184643507003784),
 ('automotive', 0.2982318699359894),
 ('intersection', 0.29432493448257446),
 ('fullresolution', 0.2853557765483856),
 ('precise', 0.2782291769981384),
 ('modern', 0.2727961540222168),
 ('drive', 0.267679899930954),
 ('scratch', 0.2633206844329834),
 ('year', 0.253436803817749),
 ('usually', 0.2403041124343872),
 ('road',

In [28]:
filt_res_2 = sort_words_by_cosine_similarity(words = keywords_2, 
                                    text = text)
filt_res_2

[('planning', 0.5416778326034546),
 ('pretraine', 0.4902573227882385),
 ('applicationsection', 0.48725610971450806),
 ('pursue', 0.44221925735473633),
 ('target', 0.43687745928764343),
 ('prove', 0.4239189624786377),
 ('novel', 0.4149957001209259),
 ('geometry', 0.3997527062892914),
 ('combine', 0.3891441226005554),
 ('navigation', 0.3869742751121521),
 ('improve', 0.3738356828689575),
 ('driver', 0.3713556230068207),
 ('output', 0.36643144488334656),
 ('start', 0.35531628131866455),
 ('self', 0.35128113627433777),
 ('yield', 0.340837299823761),
 ('accuracy', 0.3166295886039734),
 ('score', 0.3126111626625061),
 ('probability', 0.31184643507003784),
 ('goal', 0.29926344752311707),
 ('automotive', 0.2982318699359894),
 ('superior', 0.2920076847076416),
 ('train', 0.28579238057136536),
 ('fullresolution', 0.2853557765483856),
 ('modern', 0.2727961540222168),
 ('drive', 0.267679899930954),
 ('scratch', 0.2633206844329834),
 ('accurate', 0.2443581223487854),
 ('unpoole', 0.2310347557067871

In [35]:
keywords_3 = find_keywords(SentenceTokenizerSingleton().tokenizer,
                                sentences_candidat=filt_results[0], 
                                sentences_reference =filt_results[0], 
                                collocation_len=1, num_of_keywords=30) 
keywords_3

['orginal',
 'captioning',
 'minimax',
 'generation',
 'introduce',
 'work',
 'expression',
 'chief',
 'discriminative',
 'design',
 'mimic',
 'inspire',
 'recurent',
 'neural',
 'methodology',
 'generator',
 'predict',
 'generate',
 'game',
 'propose',
 'generative',
 'dialogue',
 'judge',
 'reveal',
 'text',
 'translation',
 'apply',
 'discriminator',
 'study',
 'poem']

In [36]:
filt_res_3 = sort_words_by_cosine_similarity(words = keywords_3, 
                                    text = text)
filt_res_3

[('poem', 0.32258692383766174),
 ('discriminator', 0.2925635576248169),
 ('study', 0.28695398569107056),
 ('translation', 0.2620700001716614),
 ('neural', 0.2533372938632965),
 ('dialogue', 0.2532534599304199),
 ('apply', 0.2508460283279419),
 ('text', 0.2479267716407776),
 ('mimic', 0.24710825085639954),
 ('recurent', 0.2453039288520813),
 ('reveal', 0.24464984238147736),
 ('orginal', 0.24178415536880493),
 ('judge', 0.23991701006889343),
 ('game', 0.23809026181697845),
 ('propose', 0.23602992296218872),
 ('generative', 0.23405344784259796),
 ('minimax', 0.23196937143802643),
 ('methodology', 0.23070967197418213),
 ('generator', 0.2302768975496292),
 ('discriminative', 0.2300512045621872),
 ('generate', 0.22490659356117249),
 ('design', 0.2247522473335266),
 ('inspire', 0.21164952218532562),
 ('introduce', 0.21110282838344574),
 ('work', 0.20951494574546814),
 ('expression', 0.2040991187095642),
 ('predict', 0.2033325731754303),
 ('chief', 0.203279048204422),
 ('captioning', 0.1948155

In [37]:
num_of_sample = 8

text_of_doc = df_loaded["original_text"][num_of_sample]

text_proc = TextProcesser()
sents = text_proc.split_into_sentences(text_of_doc)

sentences = [Sentence(sents[i], i) for i in range(len(sents))]

Token indices sequence length is longer than the specified maximum sequence length for this model (3538 > 512). Running this sequence through the model will result in indexing errors


In [41]:
for s in sentences:
    if s.word_token_count > 512: 
        print(s.text)
        print(s.word_token_count)

"yes "(566613, 22. 8 2%), "no "(381307, 15. 3 5%), "2 "(80031, 3. 2 2%), "1 "(46537, 1. 8 7%), "white "(41753, 1. 6 8%), "3 "(41334, 1. 6 6%), "red "(33834, 1. 3 6%), "blue "(28881, 1. 1 6%), "4 "(27174, 1. 0 9%), "green "(22453, 0. 9 %), "black "(21852, 0. 8 8%), "yellow "(17312, 0. 7 %), "brown "(14488, 0. 5 8%), "5 "(14373, 0. 5 8%), "tennis "(10941, 0. 4 4% ), "baseball "(10299, 0. 4 1%), "6 "(10103, 0. 4 1%), "orange "(9136, 0. 3 7%), "0 "(8812, 0. 3 5%), "bathroom "(8473, 0. 3 4%), "wood "(8219, 0. 3 3%), "right "(8209, 0. 3 3%), "left "(8058, 0. 3 2%), "frisbee "(7671, 0. 3 1%), "pink "(7519, 0. 3 %), "gray "(7385, 0. 3 %), "pizza "(6892, 0. 2 8%), "7 "(6005, 0. 2 4%), "kitchen "(5926, 0. 2 4%), "8 "(5592, 0. 2 3%), "cat "(5514, 0. 2 2%), "skiing "(5189, 0. 2 1%), "skateboarding "(5122, 0. 2 1%), "dog "(5092, 0. 2 1%), "snow "(4867, 0. 2 %), "black and white "(4852, 0. 2 %), "skateboard "(4697, 0. 1 9%), "surfing "(4544, 0. 1 8%), "water "(4513, 0. 1 8%), "giraffe "(4027, 0. 1 6